<div style="background:linear-gradient(135deg,#1f4e79,#2f80ed 55%,#56ccf2);color:white;padding:26px 30px;border-radius:14px;margin:6px 0"><div style="font-size:2em;font-weight:800">Did Anyone Break the PM2.5 Standard?</div><div style="font-size:1.1em;opacity:.95;margin-top:6px">A hands-on look at one year of real EPA AQS monitoring data · MARAMA Python Tutorial</div></div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">🎯 What this notebook is</div>Three blocks. Each starts from a clean slate, so you can <b>join any block even if the one before did not go to plan</b>.<ul><li><b>Block A: Get a clean table.</b> A messy raw file becomes usable data.</li><li><b>Block B: Plot it.</b> One line per site, then zoom in on a pollution episode.</li><li><b>Block C: Ask the regulatory question.</b> Hours or days? Daily or annual?</li></ul>You write the code, one small piece at a time.</div><div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ How it works</div><ul><li><b>Shift + Enter</b> runs a cell. <b>Do not use “Run All.”</b> It stops at the first cell you have not written.</li><li><b>Write it</b> cells are empty on purpose. Early ones give you most of the line; later ones give you the goal. Cells with a finished example are labeled <b>worked example</b>: run them and read them.</li><li><code>check()</code> looks at <b>what your code produced</b>. Many different lines can be right. A wrong result gets a nudge, never the answer.</li><li>Where you see <b>Predict</b>, lock in an answer <i>before</i> you run the code.</li><li><b>Stuck for more than 3 minutes?</b> Open the hint, then run the <b>checkpoint</b> cell at the top of the next block, and put <b>❓</b> in the chat so a helper knows. <b>Finished a block?</b> Put <b>✅</b> in the chat.</li></ul></div>

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0"><span style="font-size:1.3em;font-weight:800">Step 0</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Set up</span><div style="opacity:.92">Imports, the data file, and one helper file</div></div>

In [ ]:
import pandas as pd                    # data tables ("DataFrames"); think Excel, but programmable
import numpy as np                     # numbers and math
import matplotlib.pyplot as plt        # plotting
import matplotlib.dates as mdates      # tools for dates on plot axes

from pathlib import Path               # a safe way to talk about file locations

from marama_helpers import check, lock, reveal, scorecard, load_clean_table   # checking helpers (a separate file)

<p style="font-size:1.1em;margin:8px 4px"><i>Here we define the path to the data file as a string (str). A string is a piece of text between quotes.</i></p>

In [ ]:
# Look for the data file in the places it is likely to be.
# (On Colab: upload it, and marama_helpers.py, with the folder icon on the left. They land in /content.)
file_name = "AMP501_2420560-0.txt"
candidates = [
    Path("data") / file_name,            # running from the repo's main folder
    Path("..") / "data" / file_name,     # running from a sub-folder such as test/
    Path("/content") / file_name,        # Google Colab upload location
    Path("/mnt/workspace/projects/tutorials/marama_2026/data") / file_name,   # the instructor's machine
]

# next() takes the FIRST match; None is what we get if nothing was found
path_to_file = next((p for p in candidates if p.is_file()), None)

# If we did not find it, stop here and say so clearly (fail early, fail loudly)
if path_to_file is None:
    raise FileNotFoundError(f"Could not find {file_name}. Upload it, or check the folder.")

print("Found it:", path_to_file.resolve())

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0"><span style="font-size:1.3em;font-weight:800">Step ★</span><span style="font-size:1.2em;font-weight:600"> &nbsp;First, the answer we are chasing</span><div style="opacity:.92">Run this cell. You do not need to understand it yet. By the end you will have built it.</div></div>

<p style="font-size:1.1em;margin:8px 4px"><i>This is one year of hourly PM2.5 from eight monitors. Look at early June.</i></p>

In [ ]:
# WORKED EXAMPLE: run it and look. (load_clean_table builds the table you will make yourself in Block A.)
demo = load_clean_table(path_to_file)

fig, ax = plt.subplots(figsize=(10, 4.5), layout="tight")
for site in demo["Site ID"].unique():
    s = demo[demo["Site ID"] == site]
    ax.plot(s["Datetime"], s["Sample Value"], linewidth=0.7, alpha=0.5)
hourly = demo.groupby("Datetime")["Sample Value"].mean()
ax.plot(hourly.index, hourly.values, color="darkred", linewidth=2, label="Hourly mean, all sites")
ax.set_xlim(pd.Timestamp("2025-06-01"), pd.Timestamp("2025-07-01"))
ax.set_ylim(0, 120)
ax.set_ylabel("PM2.5 (ug/m3)")
ax.set_title("PM2.5 in June 2025")
ax.grid("both")
ax.legend()
plt.show()

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Predict (this one is resolved at the end of Block C)</div>The 24-hour PM2.5 standard compares a <i>daily average</i> to a limit. Looking at the spikes above: <b>did any site have a day that broke the 24-hour standard?</b> Type <code>"yes"</code> or <code>"no"</code>.</div><div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 📊 Poll</div>Vote: yes or no? <i>(Virtual: answer in the poll window. In the room: raise a hand.)</i></div>

In [ ]:
my_prediction = None      # replace None with "yes" or "no"
lock("opener", my_prediction)

<p style="font-size:1.1em;margin:8px 4px"><i>Let's go back to the raw file and find out.</i></p>

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0"><span style="font-size:1.3em;font-weight:800">Step A</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Block A: Get a clean table</span><div style="opacity:.92">From a messy raw file to data you can trust</div></div>

<p style="font-size:1.1em;margin:8px 4px"><i>Before we ask pandas to open it, let's read the first few lines as plain text.</i></p>

In [ ]:
# Open the file as plain text and print the first 3 lines (cut at 200 characters so they fit)
with open(path_to_file) as f:
    for i in range(3):
        print(f"line {i}:", f.readline()[:200])

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Look and answer</div><b>What single character separates the columns?</b> And what is unusual about lines 0 and 1?</div>

In [ ]:
separator_i_see = ""                  # the single character between column names
whats_odd_about_lines_0_and_1 = ""    # a few words is plenty

In [ ]:
check("sep", separator_i_see)
print("Your note about lines 0 and 1:", whats_odd_about_lines_0_and_1)

<p style="font-size:1.1em;margin:8px 4px"><i>First we try to naively import the file using <code>pandas.read_csv()</code>. It shouldn't work exactly as you want it.</i></p>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Predict</div>By default, <code>read_csv()</code> splits columns at <b>commas</b>. What will pandas do with this file?<br><b>A</b>: find one column &nbsp;·&nbsp; <b>B</b>: find about 28 columns &nbsp;·&nbsp; <b>C</b>: stop with an error</div><div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 📊 Poll</div>Vote: A, B or C? <i>(Virtual: answer in the poll window. In the room: raise a hand.)</i></div>

In [ ]:
my_prediction = None      # replace None with "A", "B" or "C"
lock("naive", my_prediction)

In [ ]:
# WORKED EXAMPLE: the naive read
df = pd.read_csv(path_to_file)
print(df.shape[1], "column(s)")
reveal("naive", {1: "A", 28: "B"}.get(df.shape[1], "C"))

<p style="font-size:1.1em;margin:8px 4px"><i>.read_csv() can import more delimiter types than ','. Tell it which one this file uses.</i></p>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Write it</div>Write the read again, this time telling pandas which character separates the columns. The argument is <code>sep=</code>.</div>

In [ ]:
# YOUR CODE HERE: read the file with the right separator
# df = ...


check("sep_read", df.shape[1])

<details style="margin:4px 0 10px 0"><summary style="cursor:pointer"><b>Stuck? Open a hint</b> (not the answer)</summary><ol style="margin-top:6px"><li>The pattern is <code>pd.read_csv(path_to_file, sep="...")</code>. What goes between the quotes?</li></ol></details>

<div style="border-left:7px solid #eb5757;background:rgba(235,87,87,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">⚠️ That warning, and that last row</div>The <b>DtypeWarning</b> means pandas found conflicting data types in some columns. The second header line looks like data but is text, and pandas guesses a type chunk by chunk. Also look at the last row of the table: it is a footer (<code># 60264 records were written</code>), not data.</div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💡 Worked example: three fixes in one read</div><ul><li><code>header=[0]</code>: use only the first line as the header.</li><li><code>low_memory=False</code>: read each whole column before guessing its type, so the warning goes away.</li><li><code>[:-1]</code>: a <b>slice</b>. <code>df[a:b]</code> keeps rows from <code>a</code> up to, not including, <code>b</code>. Negative numbers count from the end, so <code>[:-1]</code> keeps everything except the last row.</li></ul></div>

In [ ]:
# WORKED EXAMPLE: run it, then read the last rows and the first row
df = pd.read_csv(path_to_file, sep="|", header=[0], low_memory=False)[:-1]
print(df.shape)
df.head(2)

<div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 Discuss</div>Look at <b>row 0</b> and at the <b>Dtype</b> column from <code>df.info()</code> (run <code>df.info()</code> in a new cell if you like). Sample Value should be numbers. How could a column of numbers end up as text?<details style="margin-top:8px"><summary style="cursor:pointer"><b>Instructor's take</b> (open after you have talked)</summary><div style="margin-top:6px">Row 0 is the <b>second header line</b>, still sitting in the data. One text value turns the whole column into text. That is the same cause as the DtypeWarning. So we need to drop row 0 <i>and</i> the footer, and keep Date and Start Time as text on purpose, because we are about to glue them together.</div></details></div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Write it</div>Write the read in its <b>final form</b>: your separator, <code>header=[0]</code> and <code>low_memory=False</code>; keep <code>Date</code> and <code>Start Time</code> as <b>text</b> with <code>dtype=</code> and a dictionary; and use <b>one slice</b> to drop both the top junk row and the footer.</div>

In [ ]:
# YOUR CODE HERE: the final read
# df = ...


check("clean", (df.iloc[0]["Date"], df.shape))
df[["Date", "Start Time"]].head()

<details style="margin:4px 0 10px 0"><summary style="cursor:pointer"><b>Stuck? Open a hint</b> (not the answer)</summary><ol style="margin-top:6px"><li>A dictionary maps names to values: <code>{"Date": str, "Start Time": str}</code>. Pass it as <code>dtype=...</code>.</li><li>Row 0 is the junk row at the top, so the slice must start at 1. The footer is the last row, so it must stop at -1.</li><li>Still stuck? Run the checkpoint at the top of Block B, and put ❓ in the chat.</li></ol></details>

<p style="font-size:1.1em;margin:8px 4px"><i>To do any plotting we need this oriented by time. Right now we have no datetime information, only two strings (Date and Start Time). We can combine them into one string and then convert that to a datetime.</i></p>

In [ ]:
# WORKED EXAMPLE: glue the two text columns together, row by row:  "20250101" + "00:00"  ->  "2025010100:00"
glued = df["Date"] + df["Start Time"]
glued.head(3)

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Write it</div>Write one line that turns <code>glued</code> into a datetime column named <code>Datetime</code>, using <code>pd.to_datetime(..., format=...)</code>. Format codes: <code>%Y</code> year, <code>%m</code> month, <code>%d</code> day, <code>%H</code> hour (24-hour), <code>%M</code> minute.</div>

In [ ]:
# YOUR CODE HERE: create df["Datetime"]


check("datetime", str(df["Datetime"].iloc[0]))

<details style="margin:4px 0 10px 0"><summary style="cursor:pointer"><b>Stuck? Open a hint</b> (not the answer)</summary><ol style="margin-top:6px"><li>The shape of the call is <code>df["Datetime"] = pd.to_datetime(glued, format="...")</code>.</li><li>Write the pattern in the same order as the glued text: year, month, day, hour, a colon, minute.</li></ol></details>

<div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 Discuss</div>The file does not say what time zone these timestamps are in. Why would that matter if you wanted to line these data up with a <b>satellite overpass</b>? (Virtual: type your idea in the chat.)<details style="margin-top:8px"><summary style="cursor:pointer"><b>Instructor's take</b> (open after you have talked)</summary><div style="margin-top:6px">AQS hourly times are reported in <b>local standard time</b>, with no daylight-saving shift (see the <a href='https://aqs.epa.gov/aqsweb/documents/AQS_Format.html'>AQS format documentation</a>). Satellites report UTC, so you would need each site's offset before matching them.</div></details></div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💡 Missing data is not zero</div>Some hours have <i>no reading</i>. AQS marks them with a <b>Null Data Code</b> and leaves the value blank. A blank has to stay <b>missing</b> (NaN), or averages get dragged down and nobody knows.</div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Write it</div>Convert <code>Sample Value</code> to numbers with <code>pd.to_numeric()</code>, turning anything that is not a number into NaN. Then <b>count the hours with no reading</b> in <code>n_missing</code>. Look up <code>errors=</code>, <code>isna()</code> and <code>sum()</code>.</div>

In [ ]:
# YOUR CODE HERE: convert Sample Value to numbers, then count the missing hours
# n_missing = ...


print("Hours with no reading:", n_missing)
check("numeric", (str(df["Sample Value"].dtype), n_missing))

<details style="margin:4px 0 10px 0"><summary style="cursor:pointer"><b>Stuck? Open a hint</b> (not the answer)</summary><ol style="margin-top:6px"><li><code>errors=</code> can be <code>"raise"</code> (stop with an error), <code>"ignore"</code> (leave the text alone) or <code>"coerce"</code> (make it NaN). Which fits?</li><li><code>df["Sample Value"].isna()</code> gives True or False for each row. <code>.sum()</code> counts the Trues.</li></ol></details>

<div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 Discuss</div>The <b>Null Data Code</b> column explains the gaps. If you were writing the report, would you drop these hours, fill them in, or report them as missing? Who decides?<details style="margin-top:8px"><summary style="cursor:pointer"><b>Instructor's take</b> (open after you have talked)</summary><div style="margin-top:6px">There is no single right answer. What matters is that the choice is <b>made on purpose and written down</b>. Some AQS codes mean “the monitor ran but the data failed QA,” which is very different from “the monitor was off.”</div></details></div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ ⭐ Stretch (optional, for when you finish early)</div>Which <b>site</b> has the most missing hours, and how many? Store the site ID and the count in <code>worst_site</code> and <code>worst_count</code>.</div>

In [ ]:
# YOUR CODE HERE (optional)


check("stretch_a", (worst_site, worst_count))

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">🎯 End of Block A</div>You now have a clean table: right columns, no junk rows, real timestamps, real numbers. <b>Put ✅ in the chat.</b></div>

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0"><span style="font-size:1.3em;font-weight:800">Step B</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Block B: Plot it</span><div style="opacity:.92">One line per site, then zoom in on an episode</div></div>

<div style="border-left:7px solid #eb5757;background:rgba(235,87,87,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">⚠️ Checkpoint: start here if Block A did not work out</div>Run the next cell to load the clean table. There is no shame in it: this is what it is for. Put <b>❓</b> in the chat if you used it, so a helper can catch you up later.</div>

In [ ]:
df = load_clean_table(path_to_file)    # run this ONLY if your own table from Block A is not working

In [ ]:
# WORKED EXAMPLE: the quickest possible plot: x = time, y = the PM2.5 value
df.plot(x="Datetime", y="Sample Value");

<p style="font-size:1.1em;margin:8px 4px"><i>We got a plot, but wait... are there multiple sites in this file?</i></p>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Write it</div>Write one line that stores each distinct <code>Site ID</code> exactly once in a variable called <code>sites</code>. The pandas method's name is a word for “one of a kind.”</div>

In [ ]:
# YOUR CODE HERE: sites = ...


print(sites)
check("sites", sorted(sites))

<details style="margin:4px 0 10px 0"><summary style="cursor:pointer"><b>Stuck? Open a hint</b> (not the answer)</summary><ol style="margin-top:6px"><li>Start from the column: <code>df["Site ID"]</code>. Then add a method to it. <code>unique</code> is one; <code>drop_duplicates</code> is another.</li></ol></details>

<p style="font-size:1.1em;margin:8px 4px"><i>We should probably split things up by 'Site ID'. We could make multiple plots (but there are a lot of unique Site IDs), or we could use a legend. Let's use Matplotlib for this part...</i></p>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💡 Reading a loop out loud</div><code>for site in sites:</code> means “take the sites one at a time, call the current one <code>site</code>, and run the indented lines.” <code>df[df["Site ID"] == site]</code> means “keep only the rows where Site ID equals this site.” Two equals signs: <code>==</code> asks a question, <code>=</code> stores a value.</div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Write it</div>Write a <b>loop</b> that draws one line per site on the same plot (<code>Datetime</code> on x, <code>Sample Value</code> on y, the site as the <code>label</code>). The loop header is given. You write the body: pick out one site's rows, then call <code>plt.plot(...)</code>.</div>

In [ ]:
plt.figure()
for site in sites:
    # YOUR CODE HERE: select this site's rows, then plot them with label=site
    pass

plt.legend()
check("loop", len(plt.gca().lines))
plt.show()

<details style="margin:4px 0 10px 0"><summary style="cursor:pointer"><b>Stuck? Open a hint</b> (not the answer)</summary><ol style="margin-top:6px"><li>Inside the loop, make a smaller table: <code>df_site = df[df["Site ID"] == site]</code>.</li><li>Then plot its two columns: <code>plt.plot(df_site["Datetime"], df_site["Sample Value"], label=site)</code>.</li></ol></details>

<div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 Discuss</div>Do the sites move together or apart? Does any site have a gap or a short record? Write down what you see. (Virtual: type it in the chat.)<details style="margin-top:8px"><summary style="cursor:pointer"><b>Instructor's take</b> (open after you have talked)</summary><div style="margin-top:6px">The sites rise and fall together, so the high periods are <b>regional</b>, not one local source. Two lines are visibly shorter. We count them in Block C.</div></details></div>

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💡 Worked example: a better plot, with a daily mean</div><code>fig, ax = plt.subplots()</code> gives you named objects, and <code>ax.something()</code> says exactly <i>which</i> plot you mean. The <b>daily mean</b> groups by the calendar <i>date</i> of each timestamp (<code>.dt.date</code>), then averages across all sites.</div>

In [ ]:
# WORKED EXAMPLE: read every line, then run it
fig, ax = plt.subplots(figsize=(10, 5), layout="tight")

for site in sites:
    df_site = df[df["Site ID"] == site][["Datetime", "Sample Value"]]
    ax.plot(df_site["Datetime"], df_site["Sample Value"], label=site, linewidth=1)

# The daily mean across all sites: group by calendar date, then average
daily_mean = df.groupby(df["Datetime"].dt.date, as_index=False)["Sample Value"].mean()
ax.plot(daily_mean["Datetime"], daily_mean["Sample Value"], label="Daily mean", color="black", linewidth=2)

ax.set_ylabel("PM2.5 concentration (ug/m3)")
ax.set_xlabel("Date and time (local standard time)")
ax.set_title("PM2.5 Concentrations at Multiple Sites, 2025")
ax.set_ylim(0, 120)
ax.grid("both")
ax.xaxis.set_minor_locator(mdates.MonthLocator())
ax.legend(title="Site IDs", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.show()

<p style="font-size:1.1em;margin:8px 4px"><i>Let's do some analysis on this now. It looks like 2025-06 has some high values for several days across all sites.</i></p>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Write it</div>Zoom in on June 2025 and add the <b>hourly mean</b>. Write three pieces: the <b>hourly mean</b> (like the daily mean above, but group by the full <code>Datetime</code> column), the <b>zoom</b> with <code>ax.set_xlim(...)</code>, and a <b>y-axis label with units and a title</b>.</div>

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), layout="tight")
for site in sites:
    df_site = df[df["Site ID"] == site][["Datetime", "Sample Value"]]
    ax.plot(df_site["Datetime"], df_site["Sample Value"], label=site, linewidth=0.8, alpha=0.6)

ax.plot(daily_mean["Datetime"], daily_mean["Sample Value"], label="Daily mean", color="black", linewidth=2.5)

# YOUR CODE HERE: hourly_mean = ...   (then plot it as a dark red line, linewidth=2)


# YOUR CODE HERE: zoom the x-axis to June 2025


# YOUR CODE HERE: y-axis label (with units) and a title


check("hourly", round(float(hourly_mean["Sample Value"].sum()), 1))
lo, hi = (mdates.num2date(v).strftime("%Y-%m-%d") for v in ax.get_xlim())
check("june", f"{lo}|{hi}")
check("labels", len(ax.get_ylabel()) > 2 and len(ax.get_title()) > 2)

ax.set_ylim(0, 120)
ax.set_xlabel("Date (local standard time)")
ax.grid("both")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y-%m-%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
ax.legend(title="Lines", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.show()

<details style="margin:4px 0 10px 0"><summary style="cursor:pointer"><b>Stuck? Open a hint</b> (not the answer)</summary><ol style="margin-top:6px"><li>The hourly mean looks like the daily mean line above, but it groups by <code>"Datetime"</code> itself. Then plot it with <code>ax.plot(...)</code>.</li><li><code>ax.set_xlim(pd.Timestamp("..."), pd.Timestamp("..."))</code>. The end is the first day after June.</li><li><code>ax.set_ylabel("text")</code> and <code>ax.set_title("text")</code>. Still stuck? Run the checkpoint at the top of Block C and put ❓ in the chat.</li></ol></details>

<div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 Discuss</div>Compare the <b>red</b> line and the <b>black</b> line. Where do they disagree most? Which would you show a reporter, and which a regulator? (Virtual: chat.)<details style="margin-top:8px"><summary style="cursor:pointer"><b>Instructor's take</b> (open after you have talked)</summary><div style="margin-top:6px">The hourly mean (red) catches short spikes that the daily mean (black) averages away. Neither is “wrong.” They answer different questions, and the standards are written for a specific averaging time.</div></details></div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ ⭐ Stretch (optional, for when you finish early)</div>When is the <b>single highest hourly mean</b>? Store the timestamp in <code>peak_time</code>.</div>

In [ ]:
# YOUR CODE HERE (optional)


check("stretch_b", peak_time)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">🎯 End of Block B</div>Plots that show the episode. <b>Put ✅ in the chat.</b></div>

<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0"><span style="font-size:1.3em;font-weight:800">Step C</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Block C: Ask the regulatory question</span><div style="opacity:.92">Standards are not set on single hours</div></div>

<div style="border-left:7px solid #eb5757;background:rgba(235,87,87,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">⚠️ Checkpoint: start here if Block B did not work out</div>Run the next cell to load the clean table, then continue. Put <b>❓</b> in the chat if you used it.</div>

In [ ]:
df = load_clean_table(path_to_file)    # run this ONLY if your own table is not working

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">🎯 The question</div>The <b>24-hour PM2.5 standard</b> compares a <i>daily average</i> to a limit. So: <b>did any site have a day above it?</b> We need (1) a daily mean per site and (2) a rule for when a day has enough data to count. A common rule: at least <b>75% of the hours</b>.</div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Predict</div>Out of all the site-days, how many pass a 75% completeness rule?<br><b>"most"</b> (over 90%) &nbsp;·&nbsp; <b>"about half"</b> &nbsp;·&nbsp; <b>"few"</b> (under 50%)</div>

In [ ]:
my_prediction = None      # replace None with "most", "about half" or "few"
lock("pct", my_prediction)

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💡 Worked example: a daily table with a completeness rule</div>Group by <b>two</b> columns (site and day). <code>.agg(...)</code> with <b>named results</b> gives the mean <i>and</i> the number of valid readings. <code>count()</code> skips missing values, which is why we use it. Study this pattern: you will reuse it twice.</div>

In [ ]:
# WORKED EXAMPLE: read each line, then run it
df["Day"] = df["Datetime"].dt.date

daily = (
    df.groupby(["Site ID", "Day"])["Sample Value"]
    .agg(daily_mean="mean", valid_hours="count")   # two named results per (site, day)
    .reset_index()
)

daily["complete"] = daily["valid_hours"] >= 18     # 75% of 24 hours
daily_valid = daily[daily["complete"]]             # keep only the days that count

print(f"{len(daily)} site-days in total; {len(daily_valid)} pass the rule")
share = daily["complete"].mean()
reveal("pct", "most" if share > 0.9 else "about half" if share >= 0.5 else "few")

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💡 Reading AI-written code</div>An AI assistant suggested a slightly different rule. It runs without any error. <b>Read it, then test it.</b></div>

In [ ]:
# An AI assistant wrote this line for the completeness rule:
ai_complete = daily["valid_hours"] > 18

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Write it</div>Write code that counts <b>how many site-days get a different answer</b> under the AI's rule (<code>&gt; 18</code>) than under ours (<code>&gt;= 18</code>). Store the count in <code>n_different</code>. Then type, in your own words, whether it matters.</div>

In [ ]:
# YOUR CODE HERE: n_different = ...


my_explanation = ""     # in your own words: does it matter, and why?
check("ai_diff", n_different)
print("Your explanation:", my_explanation)

<details style="margin:4px 0 10px 0"><summary style="cursor:pointer"><b>Stuck? Open a hint</b> (not the answer)</summary><ol style="margin-top:6px"><li>Two True/False columns can be compared with <code>!=</code> (not equal). Counting the Trues is <code>.sum()</code>.</li><li><code>(ai_complete != daily["complete"]).sum()</code> is one way. Why would the two rules differ for some days and not others?</li></ol></details>

<div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 Discuss</div>Is it a big number or a small one? Does that make it unimportant? Who decides which rule is right? (Virtual: chat.)<details style="margin-top:8px"><summary style="cursor:pointer"><b>Instructor's take</b> (open after you have talked)</summary><div style="margin-top:6px">The rules only differ for days sitting <b>exactly on the boundary</b>. A small count can still change a regulatory result, and “at least 75%” means exactly 18 hours should count. A confident-looking one-character difference in generated code is exactly what careful reading catches.</div></details></div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Predict</div>Back to the opening question. Did any site-day exceed the 24-hour standard? The notebook will reveal your opening prediction when you have the answer.</div><div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 📊 Poll</div>Vote again: yes or no? Did the room change its mind? <i>(Virtual: answer in the poll window. In the room: raise a hand.)</i></div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Write it</div>Set <code>NAAQS_24HR</code> to the primary 24-hour PM2.5 standard (µg/m³), then write the code that keeps the rows of <code>daily_valid</code> with a daily mean <b>above</b> it, in <code>exceed</code>. Then print: the number of rows, the highest <b>hourly</b> value in <code>df</code>, and the highest <b>daily</b> mean.</div>

In [ ]:
# YOUR CODE HERE: NAAQS_24HR = ...
# YOUR CODE HERE: exceed = ...


check("naaqs24", NAAQS_24HR)
check("exceed", len(exceed))
reveal("opener", "yes" if len(exceed) else "no")

# YOUR CODE HERE: print the count, the highest HOURLY value, and the highest valid DAILY mean

<details style="margin:4px 0 10px 0"><summary style="cursor:pointer"><b>Stuck? Open a hint</b> (not the answer)</summary><ol style="margin-top:6px"><li>The standard is a whole number between 30 and 40. Look it up in the EPA NAAQS table.</li><li><code>daily_valid[daily_valid["daily_mean"] > NAAQS_24HR]</code> keeps the rows above the limit.</li><li><code>df["Sample Value"].max()</code> and <code>daily_valid["daily_mean"].max()</code> give the two highest values.</li></ol></details>

<div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 Discuss</div>Compare the highest <b>hourly</b> value, the highest <b>daily</b> mean, and the standard. Why can one hour be far above a limit while no <i>day</i> is? And which day has the highest daily mean? Is it where you expected?<details style="margin-top:8px"><summary style="cursor:pointer"><b>Instructor's take</b> (open after you have talked)</summary><div style="margin-top:6px">Averaging time decides the answer. <b>Always ask what number the standard actually compares.</b> The single highest daily mean (site 1301) is on <b>July 4</b>, not in June, and the highest hourly network mean falls late that night. A holiday evening is a plausible cause, but the data alone do not prove it.</div></details></div>

<div style="border-left:7px solid #eb5757;background:rgba(235,87,87,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">⚠️ A screening number, not a design value</div>The regulatory statistic for the 24-hour standard is the <b>98th percentile</b> of daily means, averaged over <b>three years</b>. We have one year, and the pandas percentile method is not the official one. Treat what follows as <b>screening</b>.</div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Write it</div>Make a table <code>site_summary</code> with one row per site: <code>valid_days</code> (count), <code>p98</code> (the 98th percentile of its daily means), and <code>max_daily</code>. Round to one decimal, sorted with the highest <code>p98</code> first. <code>quantile()</code> takes a fraction between 0 and 1. <b>Reuse the pattern from the daily-table worked example.</b></div>

In [ ]:
# YOUR CODE HERE: site_summary = ...


check("p98", site_summary["p98"].iloc[0])
site_summary

<details style="margin:4px 0 10px 0"><summary style="cursor:pointer"><b>Stuck? Open a hint</b> (not the answer)</summary><ol style="margin-top:6px"><li>Group <code>daily_valid</code> by <code>"Site ID"</code>, pick <code>"daily_mean"</code>, and use <code>.agg(...)</code> with three named results, as in the worked example.</li><li>A named result with a custom function: <code>p98=lambda s: s.quantile(...)</code>.</li><li><code>.round(1)</code> then <code>.sort_values("p98", ascending=False)</code>.</li></ol></details>

<div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 Discuss</div>Look at the <code>valid_days</code> column. What do you notice about two of the sites? Can you fairly compare them with the others? (Virtual: chat.)<details style="margin-top:8px"><summary style="cursor:pointer"><b>Instructor's take</b> (open after you have talked)</summary><div style="margin-top:6px">Sites 0067 and 1008 have far fewer valid days (the short records you may have spotted in the line plot). A low number from a site that ran for only part of the year can look great and mean nothing.</div></details></div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Predict</div>Will <b>every</b> site be below the <b>annual</b> standard? Type <code>"all below"</code> or <code>"at least one above"</code>.</div><div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 📊 Poll</div>Vote: all below, or at least one above? <i>(Virtual: answer in the poll window. In the room: raise a hand.)</i></div>

In [ ]:
my_prediction = None      # replace None with "all below" or "at least one above"
lock("over_yn", my_prediction)

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Write it</div>Set <code>NAAQS_ANNUAL</code> to the primary annual PM2.5 standard (µg/m³). Make a table <code>annual</code> with each site's <b>annual mean</b> (the average of its valid daily means), rounded to one decimal, plus a True/False column <code>above_standard</code>.</div>

In [ ]:
# YOUR CODE HERE: NAAQS_ANNUAL = ...
# YOUR CODE HERE: annual = ...


check("annual", NAAQS_ANNUAL)
check("annual_top", annual["annual_mean"].max())
reveal("over_yn", "at least one above" if annual["above_standard"].any() else "all below")
annual

<details style="margin:4px 0 10px 0"><summary style="cursor:pointer"><b>Stuck? Open a hint</b> (not the answer)</summary><ol style="margin-top:6px"><li>The annual standard is a single-digit number. Look it up in the EPA NAAQS table.</li><li>Same pattern again: group <code>daily_valid</code> by site, take the mean of <code>daily_mean</code>, name the result <code>annual_mean</code>.</li><li><code>annual["above_standard"] = annual["annual_mean"] > NAAQS_ANNUAL</code>.</li></ol></details>

<div style="border-left:7px solid #9b51e0;background:rgba(155,81,224,.13);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">💬 Discuss</div>The 24-hour check and the annual check disagree. Which kind of pollution does each standard protect against? Is a result this close to a limit a conclusion, or a reason to look closer?<details style="margin-top:8px"><summary style="cursor:pointer"><b>Instructor's take</b> (open after you have talked)</summary><div style="margin-top:6px">The 24-hour standard targets short, high episodes; the annual standard targets a persistent background. This is <b>one year</b> and a simple mean. The official annual value uses quarterly averaging, three years, strict completeness rules, and care about which instruments count.</div></details></div>

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ ⭐ Stretch (optional, for when you finish early)</div>Make a bar chart of <code>site_summary["p98"]</code> by site with the 24-hour standard as a dashed line (<code>ax.axhline</code>). Label the axes and add a title.</div>

In [ ]:
# YOUR CODE HERE (optional)



<div style="background:linear-gradient(90deg,#2f80ed,#56ccf2);color:white;padding:10px 20px;border-radius:10px;margin:22px 0 6px 0"><span style="font-size:1.3em;font-weight:800">Step ✓</span><span style="font-size:1.2em;font-weight:600"> &nbsp;Wrap-up</span><div style="opacity:.92">Say it in your own words</div></div>

In [ ]:
scorecard()   # how did your predictions go?

<div style="border-left:7px solid #f2994a;background:rgba(242,153,74,.15);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">✍️ Last exercise</div>In one or two sentences, write the most useful thing you learned. Imagine telling a colleague back at the office. (Virtual: paste it in the chat.)</div>

In [ ]:
my_takeaway = ""

In [ ]:
from IPython.display import HTML, display
display(HTML(f"<div style='border-left:7px solid #27ae60;padding:12px 18px;border-radius:8px;"
             f"background:rgba(39,174,96,.13);font-size:1.1em'><b>🎓 Your takeaway:</b><br><i>{my_takeaway}</i></div>"))

<div style="border-left:7px solid #2f80ed;background:rgba(47,128,237,.12);padding:10px 18px;margin:10px 0;border-radius:8px;line-height:1.5"><div style="font-weight:700;font-size:1.08em;margin-bottom:4px">🎯 What you practiced, and what is next</div><ul><li><b>Reading a messy file</b>, fixing it one problem at a time.</li><li><b>Checking, not trusting:</b> units, parameter code, missing data, time zone, and AI-written code.</li><li><b>Averaging time and completeness</b> change the answer. A screening result is not a design value.</li></ul>Next: <b>ozone</b> and the 8-hour average rules, <b>satellite NO₂</b> on a map next to these monitors, and a <b>pipeline</b> that runs with one command. <b>Bring one real problem:</b> what do you do every week at work that looks a bit like this? You will start automating it in the last session.</div>